<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne numpy scikit-learn

In [ ]:
import mne
import numpy as np
import os, re, subprocess, time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 1: Config

All tunable parameters. Seed=42 for reproducible split. Subset to handle RAM.

In [ ]:
BASE_DIR = "/content/drive/MyDrive/EEG_PROJECT"
os.makedirs(BASE_DIR, exist_ok=True)

# 11 patients, serial order, shuffled with seed=42
ALL_PATIENTS = [f"chb{i:02d}" for i in range(1, 12)]
np.random.seed(42)
shuffled = ALL_PATIENTS.copy()
np.random.shuffle(shuffled)
TRAIN_PATIENTS = sorted(shuffled[:9])
TEST_PATIENTS  = sorted(shuffled[9:])

print(f"All {len(ALL_PATIENTS)} patients: {ALL_PATIENTS}")
print(f"Train ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}")
print(f"Test  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}")

# Pipeline params
WINDOW_SIZE_SEC = 7.0
OVERLAP         = 0.3
LOWCUT          = 0.5
HIGHCUT         = 40.0
NOTCH_FREQ      = 60.0

# Balancing
NON_SEIZURE_SAMPLES  = 500
RANDOM_SEED          = 42

# Subset mode: process fewer recordings for testing (set to None for full run)
MAX_RECORDINGS_PER_PATIENT = None  # None = all | 3 = first 3 recordings per patient

print(f"Window: {WINDOW_SIZE_SEC}s | Overlap: {OVERLAP} | Filter: {LOWCUT}-{HIGHCUT} Hz")
print(f"Non-seizure samples: {NON_SEIZURE_SAMPLES} | Max rec/patient: {MAX_RECORDINGS_PER_PATIENT or 'ALL'}")

## Cell 2: Download Data

6 parallel downloads. Skips files already on Drive.

In [ ]:
start_time = time.time()
total_bytes = 0

for i in range(1, 12):
    pid = f"chb{i:02d}"
    pdir = os.path.join(BASE_DIR, pid)
    os.makedirs(pdir, exist_ok=True)

    # Download summary
    summary_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/{pid}-summary.txt"
    summary_path = os.path.join(BASE_DIR, f"{pid}-summary.txt")
    if not os.path.exists(summary_path):
        subprocess.run(["wget", "-q", "-O", summary_path, summary_url])
    total_bytes += os.path.getsize(summary_path)

    # Download RECORDS
    records_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/RECORDS"
    records_path = os.path.join(pdir, "RECORDS")
    subprocess.run(["wget", "-q", "-O", records_path, records_url])

    edf_list = []
    try:
        with open(records_path, "r") as f:
            edf_list = [l.strip() for l in f.read().splitlines() if l.strip().endswith(".edf")]
    except:
        pass

    source = "RECORDS"
    if len(edf_list) < 5:
        try:
            with open(summary_path, "r") as f:
                content = f.read()
            edf_list = re.findall(r"File Name:\s*(\S+\.edf)", content)
            source = "summary"
        except:
            pass

    base_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/"
    tasks = [(f.strip(), os.path.join(pdir, f.strip()), base_url) for f in edf_list if f.strip()]

    print(f"\n{pid}: {len(tasks)} files ({source}) [6 parallel]")

    def dl_one(args):
        fn, epath, ub = args
        nw = False
        if not os.path.exists(epath):
            subprocess.run(["wget","-q","-O",epath, ub+fn],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            nw = True
        sz = os.path.getsize(epath) / (1024*1024)
        sp = epath + ".seizures"
        if not os.path.exists(sp):
            subprocess.run(["wget","-q","-O",sp, ub+fn+".seizures"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return fn, sz, nw

    done = 0; nc = 0
    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [pool.submit(dl_one, t) for t in tasks]
        for f in as_completed(futures):
            fn, sz, is_new = f.result()
            done += 1
            m = " NEW" if is_new else ""
            if is_new: nc += 1
            print(f"  [{done}/{len(tasks)}] {fn} ({sz:.0f} MB){m}")

    elapsed = time.time() - start_time
    pbytes = sum(os.path.getsize(os.path.join(pdir, f)) for f in edf_list)
    total_bytes += pbytes
    print(f"  {nc} new, {done-nc} existed | Total: {total_bytes/1e9:.1f} GB | {elapsed/60:.0f} min")

print(f"\n===== DONE ({total_bytes/1e9:.2f} GB in {(time.time()-start_time)/60:.0f} min) =====")

## Cell 2b: Add More Patients

Downloads 3 additional patients (chb12, chb13, chb14) without re-running the full download.
Run this independently when you add new patients.

In [ ]:
EXTRA_PATIENTS = ["chb12", "chb13", "chb14"]

for pid in EXTRA_PATIENTS:
    pdir = os.path.join(BASE_DIR, pid)
    os.makedirs(pdir, exist_ok=True)

    # Download summary
    summary_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/{pid}-summary.txt"
    summary_path = os.path.join(BASE_DIR, f"{pid}-summary.txt")
    if not os.path.exists(summary_path):
        subprocess.run(["wget", "-q", "-O", summary_path, summary_url])

    # Download RECORDS
    records_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/RECORDS"
    records_path = os.path.join(pdir, "RECORDS")
    subprocess.run(["wget", "-q", "-O", records_path, records_url])

    edf_list = []
    try:
        with open(records_path, "r") as f:
            edf_list = [l.strip() for l in f.read().splitlines() if l.strip().endswith(".edf")]
    except:
        pass

    source = "RECORDS"
    if len(edf_list) < 5:
        try:
            with open(summary_path, "r") as f:
                content = f.read()
            edf_list = re.findall(r"File Name:\\s*(\\S+\\.edf)", content)
            source = "summary"
        except:
            pass

    base_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/"
    tasks = [(f.strip(), os.path.join(pdir, f.strip()), base_url) for f in edf_list if f.strip()]

    print(f"\\n{pid}: {len(tasks)} files ({source}) [6 parallel]")

    def dl_one(args):
        fn, epath, ub = args
        nw = False
        if not os.path.exists(epath):
            subprocess.run(["wget","-q","-O",epath, ub+fn],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            nw = True
        sz = os.path.getsize(epath) / (1024*1024)
        sp = epath + ".seizures"
        if not os.path.exists(sp):
            subprocess.run(["wget","-q","-O",sp, ub+fn+".seizures"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return fn, sz, nw

    done = 0; nc = 0
    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [pool.submit(dl_one, t) for t in tasks]
        for f in as_completed(futures):
            fn, sz, is_new = f.result()
            done += 1
            m = " NEW" if is_new else ""
            if is_new: nc += 1
            print(f"  [{done}/{len(tasks)}] {fn} ({sz:.0f} MB){m}")

    print(f"  {nc} new, {done-nc} existed")

print(f"\\nDone: added {len(EXTRA_PATIENTS)} patients.")

## Cell 3: Parse Seizure Annotations

Reads summary.txt files. No hardcoded timestamps.

In [ ]:
def parse_seizure_summary(summary_path):
    with open(summary_path, "r") as f:
        content = f.read()
    sections = re.split(r"File Name:\s*", content)[1:]
    sm = {}
    for s in sections:
        m = re.match(r"(\S+)", s)
        if not m: continue
        fn = m.group(1)
        st = [int(x) for x in re.findall(r"Seizure\s+Start\s+Time:\s*(\d+)\s*seconds", s)]
        en = [int(x) for x in re.findall(r"Seizure\s+End\s+Time:\s*(\d+)\s*seconds", s)]
        sm[fn] = list(zip(st, en))
    return sm

SEIZURE_MAP = {}
for p in ALL_PATIENTS:
    sp = os.path.join(BASE_DIR, f"{p}-summary.txt")
    if os.path.exists(sp):
        SEIZURE_MAP[p] = parse_seizure_summary(sp)
        n_seiz = sum(1 for v in SEIZURE_MAP[p].values() if v)
        print(f"{p}: {len(SEIZURE_MAP[p])} recordings, {n_seiz} with seizures")
    else:
        SEIZURE_MAP[p] = {}

## Cell 4: bandpass_filter()

Filter full recording ONCE before windowing. No per-window filtering.

In [ ]:
def bandpass_filter(raw, lowcut=0.5, highcut=40.0, notch_freq=60.0):
    rf = raw.copy()
    rf.filter(lowcut, highcut, fir_design="firwin", verbose=False)
    rf.notch_filter(freqs=notch_freq, verbose=False)
    return rf.get_data()

## Cell 5: create_windows()

30% overlap -> stride=1254 samples (7s @ 256Hz).

In [ ]:
def create_windows(signal, sfreq, window_size_sec=7.0, overlap=0.3):
    ws = int(window_size_sec * sfreq)
    st = int(ws * (1 - overlap))
    total = signal.shape[1]
    wins = []
    for s in range(0, total - ws, st):
        wins.append(signal[:, s:s+ws])
    return np.array(wins)

## Cell 6: create_labels()

Soft overlap: any part of window touches seizure -> label=1.

In [ ]:
def create_labels(windows, sfreq, seizure_intervals):
    n = windows.shape[0]
    ws = windows.shape[2]
    stride = int(ws * (1 - 0.3))
    labs = np.zeros(n, dtype=int)
    if not seizure_intervals: return labs
    for i in range(n):
        a = (i*stride)/sfreq
        b = (i*stride+ws)/sfreq
        for st, en in seizure_intervals:
            if b >= st and a <= en:
                labs[i] = 1; break
    return labs

## Cell 7: process_recording()

Standardizes to 23 EEG channels. Crash-free: skips bad files.

In [ ]:
# Standard 23-channel subset (bipolar EEG only, skip EKG/EMG refs)
EEG_23_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ",
    "P7-T7", "T7-FT9", "FT9-FT10", "FT10-T8", "T8-P8"
]

def process_recording(edf_path, seizure_intervals,
                      window_size_sec=7.0, overlap=0.3,
                      lowcut=0.5, highcut=40.0, notch_freq=60.0):
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    except Exception as e:
        print(f"    SKIP {os.path.basename(edf_path)}: {e}")
        return None, None

    # Standardize to 23 channels (pick channels that exist in this recording)
    available = raw.ch_names
    channels_to_use = [ch for ch in EEG_23_CHANNELS if ch in available]
    if len(channels_to_use) < 18:
        print(f"    SKIP {os.path.basename(edf_path)}: only {len(channels_to_use)} matching channels")
        return None, None

    raw.pick_channels(channels_to_use)
    n_chan = len(channels_to_use)
    channel_names = channels_to_use.copy()

    signal = bandpass_filter(raw, lowcut, highcut, notch_freq)
    sfreq = raw.info["sfreq"]
    windows = create_windows(signal, sfreq, window_size_sec, overlap)

    # Pad to 23 channels if needed (zero-fill missing channels)
    if n_chan < 23:
        pad = np.zeros((23 - n_chan, windows.shape[1], windows.shape[2]), dtype=windows.dtype)
        windows = np.concatenate([windows, pad], axis=0)
        # Mark padded channels as "---MISSING---"
        channel_names += ["---MISSING---"] * (23 - n_chan)
    elif n_chan > 23:
        windows = windows[:23]
        channel_names = channel_names[:23]


    # Skip recordings too short for even one window
    if windows.ndim != 3 or windows.shape[0] == 0:
        print(f"    SKIP {os.path.basename(edf_path)}: too short")
        return None, None, None

    labels = create_labels(windows, sfreq, seizure_intervals)
    return windows, labels, channel_names

## Cell 8: preprocess_streaming() -- RAM-Safe

Processes recordings one at a time. Reservoir-samples non-seizure windows.
Never holds more than 500 + seizure_count windows in RAM.

In [ ]:
def preprocess_streaming(patient_ids, seizure_map, base_dir,
                         n_non_seizure=500, seed=42,
                         max_rec=None):
    np.random.seed(seed)

    all_seizure_wins = []
    all_seizure_labs = []

    # Reservoir for non-seizure (pre-allocated, never grows)
    non_wins = None  # allocated on first non-seizure window
    non_labs = np.zeros(n_non_seizure, dtype=int)
    non_count = 0
    n_chan_expected = None

    total_recordings = 0
    skipped = 0

    for pid in patient_ids:
        pdir = os.path.join(base_dir, pid)
        if not os.path.isdir(pdir):
            continue

        edf_files = sorted([f for f in os.listdir(pdir) if f.endswith(".edf")])
        if max_rec: edf_files = edf_files[:max_rec]

        for edf in edf_files:
            intervals = seizure_map[pid].get(edf, [])
            path = os.path.join(pdir, edf)

            result = process_recording(path, intervals,
                                       window_size_sec=WINDOW_SIZE_SEC,
                                       overlap=OVERLAP)
            if result[0] is None:
                skipped += 1
                continue

            windows, labels, ch_names = result
            total_recordings += 1

            # Lazy-allocate non-seizure reservoir
            if non_wins is None:
                n_chan = windows.shape[1]
                non_wins = np.zeros((n_non_seizure, n_chan, windows.shape[2]), dtype=np.float32)

            # Split seizure vs non-seizure
            sez_mask = labels == 1
            non_mask = labels == 0

            # Keep ALL seizure windows
            if sez_mask.any():
                all_seizure_wins.append(windows[sez_mask])
                all_seizure_labs.append(labels[sez_mask])

            # Reservoir sample non-seizure windows
            non_idx = np.where(non_mask)[0]
            for idx in non_idx:
                non_count += 1
                if non_count <= n_non_seizure:
                    non_wins[non_count - 1] = windows[idx]
                else:
                    j = np.random.randint(0, non_count)
                    if j < n_non_seizure:
                        non_wins[j] = windows[idx]

            n_s = int(sez_mask.sum())
            n_n = int(non_mask.sum())
            print(f"  {pid}/{edf} -> {windows.shape[0]} windows ({n_s} sez, {n_n} non) | "
                  f"total sez: {sum(w.shape[0] for w in all_seizure_wins)} | "
                  f"reservoir: {min(non_count, n_non_seizure)}")

    # Combine
    if all_seizure_wins:
        sez_win = np.concatenate(all_seizure_wins, axis=0)
        sez_lab = np.concatenate(all_seizure_labs, axis=0)
    else:
        sez_win = np.zeros((0, non_wins.shape[1], non_wins.shape[2]), dtype=np.float32)
        sez_lab = np.zeros(0, dtype=int)

    X = np.concatenate([sez_win, non_wins], axis=0)
    y = np.concatenate([sez_lab, non_labs], axis=0)
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]

    print(f"\nDone: {total_recordings} recordings, {skipped} skipped")
    print(f"  Seizure windows: {len(sez_win)}")
    print(f"  Non-seizure windows: {min(non_count, n_non_seizure)} (from {non_count} total)")
    print(f"  Final shape: X={X.shape}, y={y.shape}")
    return X.astype(np.float32), y, ch_names

## Cell 9: Preprocess Training Set

Streaming processing. 9 patients. Output stays in RAM (balanced to ~500+seizure).

In [ ]:
print("\n===== PREPROCESSING TRAIN SET =====")
print(f"Patients: {TRAIN_PATIENTS}\n")

X_train, y_train, ch_names = preprocess_streaming(
    TRAIN_PATIENTS, SEIZURE_MAP, BASE_DIR,
    n_non_seizure=NON_SEIZURE_SAMPLES, seed=RANDOM_SEED,
    max_rec=MAX_RECORDINGS_PER_PATIENT)

print(f"\nTrain ready: {X_train.shape}, {y_train.shape}")
print(f"Channels: {ch_names}")
print(f"Seizure: {np.sum(y_train)}, Non-seizure: {len(y_train)-np.sum(y_train)}")

## Cell 10: Preprocess Test Set

Streaming processing. 2 held-out patients.

In [ ]:
print("\n===== PREPROCESSING TEST SET =====")
print(f"Patients: {TEST_PATIENTS}\n")

X_test, y_test, _ = preprocess_streaming(
    TEST_PATIENTS, SEIZURE_MAP, BASE_DIR,
    n_non_seizure=NON_SEIZURE_SAMPLES, seed=RANDOM_SEED,
    max_rec=MAX_RECORDINGS_PER_PATIENT)

print(f"\nTest ready: {X_test.shape}, {y_test.shape}")
print(f"Seizure: {np.sum(y_test)}, Non-seizure: {len(y_test)-np.sum(y_test)}")

## Cell 11: Normalize Per-Channel

Each of the 23 EEG channels normalized independently.
Stats computed from TRAIN data only, applied to both train and test.

In [ ]:
# Compute per-channel mean/std from TRAIN only
train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std  = X_train.std( axis=(0, 2), keepdims=True)

X_train = (X_train - train_mean) / (train_std + 1e-8)
X_test  = (X_test  - train_mean) / (train_std + 1e-8)

print(f"Normalized per-channel.")
print(f"  Train shape: {X_train.shape}  (samples, channels, time)")
print(f"  Test  shape: {X_test.shape}")
print(f"  Channel means after norm: {X_train.mean(axis=(0,2))[:5]}...")

## Cell 12: Save Preprocessed Data to Drive

Saves .npy files + metadata (patient split, channel names).
Ready for upload to Kaggle Datasets.

In [ ]:
OUTPUT_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save .npy files
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"),  y_test)

# Save metadata
with open(os.path.join(OUTPUT_DIR, "info.txt"), "w") as f:
    f.write(f"Train patients ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}\n")
    f.write(f"Test patients  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}\n")
    f.write(f"Window: {WINDOW_SIZE_SEC}s, Overlap: {OVERLAP}\n")
    f.write(f"Non-seizure samples: {NON_SEIZURE_SAMPLES}\n")
    f.write(f"Random seed: {RANDOM_SEED}\n")
    f.write(f"Max recordings per patient: {MAX_RECORDINGS_PER_PATIENT or 'ALL'}\n")
    f.write(f"Train shape: {X_train.shape}\n")
    f.write(f"Test shape:  {X_test.shape}\n")
    f.write(f"Channels ({len(ch_names)}): {ch_names}\n")

# Print file sizes
for fn in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, fn))
    print(f"  {fn}: {sz/1e6:.1f} MB")

total = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f))
            for f in os.listdir(OUTPUT_DIR))
print(f"\nTotal: {total/1e6:.1f} MB saved to:")
print(f"  {OUTPUT_DIR}/")
print(f"\nNEXT: Download from Drive -> Upload to Kaggle Datasets")